In [ ]:
import pika

def create_queue_pair(base_pattern: str):
    """
    Cria duas filas quorum:
      - NORM (principal) com DLX configurado
      - DLX (fila de dead-letter), também do tipo quorum
    E um exchange DLX (direct) ligado à fila DLX.

    :param base_pattern: string com {argument}, ex: 'G.{argument}.PROD.ActiveCampaign'
    """

    # Nomes
    norm_queue = base_pattern.replace("{argument}", "NORM")
    dlx_queue = base_pattern.replace("{argument}", "DLX")
    dlx_exchange = dlx_queue  # mesmo nome para o exchange DLX

    # Conexão
    credentials = pika.PlainCredentials("i92tech", "RHTebWoJR8xioIKCsoxw7XmyPkqafNGk0vr4IHSc")
    params = pika.ConnectionParameters(
        host="116.203.250.171",
        port=5672,
        virtual_host="default",
        credentials=credentials
    )

    connection = pika.BlockingConnection(params)
    channel = connection.channel()

    # 1) Exchange DLX (direct, durável)
    channel.exchange_declare(
        exchange=dlx_exchange,
        exchange_type="direct",
        durable=True
    )

    # 2) Fila DLX como QUORUM
    channel.queue_declare(
        queue=dlx_queue,
        durable=True,
        arguments={
            "x-queue-type": "quorum"
        }
    )

    # 3) Bind exchange DLX -> fila DLX
    channel.queue_bind(
        exchange=dlx_exchange,
        queue=dlx_queue,
        routing_key="dlx"
    )

    # 4) Fila principal NORM como QUORUM, apontando para o DLX
    channel.queue_declare(
        queue=norm_queue,
        durable=True,
        arguments={
            "x-queue-type": "quorum",
            "x-dead-letter-exchange": dlx_exchange,
            "x-dead-letter-routing-key": "dlx"
        }
    )

    print(
        "✅ Filas criadas com sucesso:"
        f"\n- {norm_queue} (quorum, com DLX)"
        f"\n- {dlx_queue} (quorum, DLX)"
        f"\n⚙️  Exchange DLX: {dlx_exchange} (direct)"
    )

    connection.close()


In [ ]:
import pika
from pika.exceptions import ChannelClosedByBroker

def _connect():
    credentials = pika.PlainCredentials("i92tech", "RHTebWoJR8xioIKCsoxw7XmyPkqafNGk0vr4IHSc")
    params = pika.ConnectionParameters(
        host="rabbitmq-manager1.i92tecnologia.com.br",
        port=5672,
        virtual_host="default",
        credentials=credentials
    )
    return pika.BlockingConnection(params)

def delete_queue_pair(base_pattern: str, *, verbose: bool = True):
    """
    Deleta a fila quorum (NORM), a fila DLX e o exchange DLX criados por create_queue_pair.

    :param base_pattern: string com {argument}, ex: 'G.{argument}.PROD.ActiveCampaign'
    :param verbose: imprime logs simples
    """
    norm_queue = base_pattern.replace("{argument}", "NORM")
    dlx_queue = base_pattern.replace("{argument}", "DLX")
    dlx_exchange = dlx_queue  # mesmo nome usado na criação

    conn = _connect()
    try:
        ch = conn.channel()

        # 1) Deletar fila NORM (quorum)
        try:
            ch.queue_delete(queue=norm_queue)
            if verbose:
                print(f"Removida fila NORM: {norm_queue}")
        except ChannelClosedByBroker as e:
            # Reabre canal se o broker fechou por 404/inelegível
            if e.reply_code == 404:
                if verbose:
                    print(f"Fila NORM não existe: {norm_queue}")
                ch = conn.channel()
            else:
                raise

        # 2) Deletar fila DLX (clássica)
        try:
            ch.queue_delete(queue=dlx_queue)
            if verbose:
                print(f"Removida fila DLX: {dlx_queue}")
        except ChannelClosedByBroker as e:
            if e.reply_code == 404:
                if verbose:
                    print(f"Fila DLX não existe: {dlx_queue}")
                ch = conn.channel()
            else:
                raise

        # 3) Deletar exchange DLX
        try:
            # if_unused=False garante deleção mesmo se o broker achar bindings pendentes
            ch.exchange_delete(exchange=dlx_exchange, if_unused=False)
            if verbose:
                print(f"Removido exchange DLX: {dlx_exchange}")
        except ChannelClosedByBroker as e:
            if e.reply_code == 404:
                if verbose:
                    print(f"Exchange DLX não existe: {dlx_exchange}")
            else:
                raise

        if verbose:
            print("Concluído: filas e exchange deletados (ou já inexistentes).")

    finally:
        conn.close()

# Exemplo de uso:
# delete_queue_pair('G.{argument}.PROD.ActiveCampaign')


In [ ]:
QUEUES = [
    "G.{argument}.PROD.Hotmart",
    #"G.{argument}.PROD.ActiveCampaign",
    "G.{argument}.PROD.Moskit_Atendimento_Sync",
    "N.{argument}.PROD",
    "NF.{argument}.PROD",
    "S.{argument}.PROD.NEW",
    "CRM.{argument}.PROD",
    "C.{argument}.PROD",
    "O.{argument}.PROD",
    "DELAY.{argument}.PROD",
    "X.{argument}.PROD.BFA25-Onboarding-utility-alunos",
    "X.{argument}.PROD.BF25-Onboarding-utility",
    "X.{argument}.PROD.lpm_approach",
    "X.{argument}.PROD.Moskit",
    "X.{argument}.PROD.Unnichat",
    "X.{argument}.PROD.CustomServices"
]

QUEUES = [{"name": queue, "status": False} for queue in QUEUES]

In [ ]:
for queue in QUEUES:
    create_queue_pair(queue.get('name'))


In [ ]:
create_queue_pair("X.{argument}.PROD.Omnichat")